# Comparación de líneas base · GPU T4
Subir la carpeta partes_v2 con Upload to Colab. Ejecutar todas las celdas. Las huellas verifican la reconstrucción de ambos ZIP.

In [3]:
from pathlib import Path
import sys, hashlib, zipfile, json, csv
import torch
assert Path('/content').is_dir() and torch.cuda.is_available(), 'Se requiere un kernel GPU de Colab'
print(torch.cuda.get_device_name(0))
raiz=Path('/content/imo_v2')
salida=Path('/content/IMO_resultados_v2')
def extraer(nombre, huella):
    import time
    dst=Path('/content')/nombre
    if not dst.exists() or hashlib.sha256(dst.read_bytes()).hexdigest()!=huella:
        for intento in range(240):
            trozos=sorted(Path('/content').rglob(Path(nombre).stem+'_*.bin'))
            if sum(f.stat().st_size for f in trozos)=={'imo_colab_train_val.zip':184858955,'test_v2.zip':33081044}[nombre]:
                tmp=dst.with_suffix('.partial')
                with tmp.open('wb') as out:
                    for f in trozos: out.write(f.read_bytes())
                if hashlib.sha256(tmp.read_bytes()).hexdigest()==huella:
                    tmp.replace(dst)
                    break
            if intento % 6 == 0: print('Verificando transferencia:',nombre,len(trozos),flush=True)
            time.sleep(5)
        else: raise RuntimeError('Transferencia incompleta; verificar partes en /content')
    archivos=[p for p in {Path.cwd()/nombre,Path('/content')/nombre} if p.is_file()]
    assert archivos, f'Subir {nombre} con Upload to Colab'
    p=archivos[0]
    assert hashlib.sha256(p.read_bytes()).hexdigest()==huella, f'Paquete distinto: {nombre}'
    with zipfile.ZipFile(p) as z: z.extractall(raiz)
extraer('imo_colab_train_val.zip', '13045ae69c3cdfc60259d722624c258e57586737568df884e562fdacde42fedf')
scripts=raiz/'tesis_imo_para_codex/scripts'
sys.path.insert(0,str(scripts))


Tesla T4


In [ ]:
codigo='"""Adaptaciones compactas para segmentación binaria, sin pesos externos."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom modelos_base import block\n\nclass Piramide(nn.Module):\n    def __init__(self, modo):\n        super().__init__()\n        self.encoder = nn.Sequential(block(3,16), nn.MaxPool2d(2), block(16,32),\n                                     nn.MaxPool2d(2), block(32,64))\n        self.modo = modo\n        if modo == \'aspp\':\n            self.ramas = nn.ModuleList([nn.Conv2d(64,16,1)] + [\n                nn.Conv2d(64,16,3,padding=d,dilation=d) for d in (2,4,8)] +\n                [nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(64,16,1))])\n            canales = 80\n        else:\n            self.ramas = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(b),\n                           nn.Conv2d(64,16,1)) for b in (1,2,3,6)])\n            canales = 128\n        self.head = nn.Sequential(nn.Conv2d(canales,32,1),nn.ReLU(),nn.Conv2d(32,1,1))\n    def forward(self,x):\n        h = self.encoder(x)\n        partes = [h] if self.modo == \'psp\' else []\n        for rama in self.ramas:\n            y = F.relu(rama(h))\n            if y.shape[-2:] != h.shape[-2:]:\n                y = F.interpolate(y,size=h.shape[-2:],mode=\'bilinear\',align_corners=False)\n            partes.append(y)\n        return F.interpolate(self.head(torch.cat(partes,1)),size=x.shape[-2:],\n                             mode=\'bilinear\',align_corners=False)\nBASES = {\'deeplab_compacta\': lambda: Piramide(\'aspp\'), \'pspnet_compacta\': lambda: Piramide(\'psp\')}\n\n'
(scripts/'bases_v2.py').write_text(codigo, encoding='utf8')
from bases_v2 import BASES
runner_source='"""Piloto portable: train/val únicamente, checkpoints por época y semillas comunes.\n\nNo modifica particiones ni accede al test. Las adaptaciones no son réplicas de\nlos artículos. Las advertencias de determinismo se conservan en el cuaderno:\nsemillas no garantizan igualdad bit a bit entre equipos/versiones.\n"""\nimport hashlib\nimport json\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport PIL\nimport torch\nfrom dataset_ready import load_example, metrics, partition\nfrom modelos_base import MODELS\n\n\ndef environment():\n    return {\'torch\': str(torch.__version__), \'numpy\': np.__version__,\n            \'pillow\': PIL.__version__, \'cuda\': torch.version.cuda,\n            \'device\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}\n\n\ndef setup(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(2)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    # Algunas operaciones (p. ej. unpooling) dependen del backend. No ocultar\n    # advertencias ni prometer determinismo estricto cuando no esté disponible.\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef tensors(row, device):\n    x, y, valid = load_example(row)\n    return (torch.from_numpy(x.copy())[None].to(device),\n            torch.from_numpy(y.astype(np.float32))[None, None].to(device),\n            torch.from_numpy(valid.copy())[None, None].to(device))\n\n\ndef loss_fn(logits, target, valid):\n    if logits.shape != target.shape:\n        raise ValueError(\'La salida no coincide con la máscara\')\n    p, t = torch.sigmoid(logits)[valid], target[valid]\n    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits[valid], t)\n    dice = 1 - (2 * (p*t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)\n    return (bce + dice) / 2\n\n\ndef sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize()\n\n\ndef smoke(output):\n    """Cinco pasos/modelo; prueba funcional, NO medición de calidad predictiva."""\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    rows = partition(\'train\')[:5]\n    results = []\n    for name, constructor in MODELS.items():\n        setup(11)\n        model = constructor().to(device)\n        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)\n        if device.type == \'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n        times = []\n        for row in rows:\n            sync(device); start = time.perf_counter()\n            x, y, v = tensors(row, device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(x), y, v)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Pérdida no finita\')\n            loss.backward(); optimizer.step()\n            sync(device); times.append(time.perf_counter() - start)\n        results.append({\'model\': name, \'steps\': len(times), \'seconds_per_step\': times,\n                        \'parameters\': sum(p.numel() for p in model.parameters()),\n                        \'peak_gpu_allocated_mib\': torch.cuda.max_memory_allocated()/2**20\n                        if device.type == \'cuda\' else None})\n        print(results[-1], flush=True)\n        del model, optimizer, x, y, v, loss\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    report = {\'environment\': environment(), \'results\': results, \'seed\': 11,\n              \'quality_metrics\': \'No se evalúan: son cinco pasos de prueba\'}\n    (output/\'smoke.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    return report\n\n\n@torch.no_grad()\ndef evaluate(model, rows, device, guard=None):\n    model.eval(); grouped = defaultdict(list); detail = []\n    for row in rows:\n        if guard:\n            guard()\n        x, y, v = tensors(row, device)\n        scores = metrics(torch.sigmoid(model(x))[0, 0].cpu().numpy(),\n                         y[0, 0].cpu().numpy(), v[0, 0].cpu().numpy())\n        grouped[row[\'group_id\']].append(scores[\'Dice\'])\n        detail.append({\'id\': row[\'id\'], \'group_id\': row[\'group_id\'], **scores})\n    return float(np.mean([np.mean(values) for values in grouped.values()])), detail\n\n\ndef run(name, seed, output, epochs=30, lr=0.001, patience=5,\n        identity=None, guard=None, max_epochs_this_call=None, force_cpu=False):\n    """Reanuda al final de la última época guardada; igual presupuesto/modelo.\n\n    Un corte dentro de una época repite esa época desde el checkpoint anterior.\n    Cargar checkpoints solo de esta ejecución privada: torch.load puede ejecutar\n    código cuando weights_only=False. No aceptar checkpoints de terceros.\n    """\n    if name not in MODELS or epochs < 1:\n        raise ValueError(\'Configuración inválida\')\n    setup(seed)\n    device = torch.device(\'cpu\' if force_cpu else (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n    root = Path(__file__).resolve().parents[1]\n    # La identidad incluye todos los archivos del paquete, por tanto código,\n    # particiones, imágenes y máscaras. Cambios rechazan una reanudación silenciosa.\n    package_digest = identity or hashlib.sha256((root.parent/\'package_manifest.json\').read_bytes()).hexdigest()\n    config = {\'model\': name, \'seed\': seed, \'epochs\': epochs, \'lr\': lr,\n              \'patience\': patience, \'batch_size\': 1, \'size\': 256,\n              \'package_manifest_sha256\': package_digest, \'environment\': environment(),\n              \'execution_device\': str(device)}\n    dest = Path(output)/f\'{name}_seed{seed}_lr{lr:g}\'\n    dest.mkdir(parents=True, exist_ok=True)\n    cfg = dest/\'config.json\'\n    if cfg.exists() and json.loads(cfg.read_text()) != config:\n        raise ValueError(\'Cambió configuración, datos o entorno. Usar otra carpeta de resultados.\')\n    cfg.write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\n    model = MODELS[name]().to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n    train, val = partition(\'train\'), partition(\'val\')\n    start_epoch, best, stale, history = 0, -1.0, 0, []\n    checkpoint = dest/\'last.pt\'\n    if checkpoint.exists():\n        saved = torch.load(checkpoint, map_location=\'cpu\', weights_only=False)\n        model.load_state_dict(saved[\'model\']); optimizer.load_state_dict(saved[\'optimizer\'])\n        start_epoch, best, stale, history = saved[\'epoch\']+1, saved[\'best\'], saved[\'stale\'], saved[\'history\']\n        torch.set_rng_state(saved[\'torch_rng\'])\n        if device.type == \'cuda\' and saved[\'cuda_rng\'] is not None:\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n    end_epoch = epochs if max_epochs_this_call is None else min(epochs, start_epoch+max_epochs_this_call)\n    for epoch in range(start_epoch, end_epoch):\n        if stale >= patience:\n            break\n        model.train(); order = list(train)\n        random.Random(seed * 100000 + epoch).shuffle(order)\n        sync(device); start = time.perf_counter(); total_loss = 0\n        for row in order:\n            if guard:\n                guard()\n            x, y, v = tensors(row, device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(x), y, v)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Pérdida no finita\')\n            loss.backward(); optimizer.step(); total_loss += loss.item()\n        score, details = evaluate(model, val, device, guard=guard)\n        improved = score > best\n        if improved:\n            best, stale = score, 0\n            torch.save(model.state_dict(), dest/\'best.tmp\')\n            os.replace(dest/\'best.tmp\', dest/\'best.pt\')\n            (dest/\'best_val_predictions.json\').write_text(json.dumps(details), encoding=\'utf-8\')\n        else:\n            stale += 1\n        sync(device)\n        history.append({\'epoch\': epoch+1, \'train_loss\': total_loss/len(train),\n                        \'val_group_Dice\': score, \'seconds\': time.perf_counter()-start})\n        torch.save({\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                    \'epoch\': epoch, \'best\': best, \'stale\': stale, \'history\': history,\n                    \'torch_rng\': torch.get_rng_state(),\n                    \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else None},\n                   dest/\'last.tmp\')\n        os.replace(dest/\'last.tmp\', checkpoint)\n        (dest/\'history.json\').write_text(json.dumps(history, indent=2), encoding=\'utf-8\')\n        print(name, seed, history[-1], flush=True)\n    result = {\'model\': name, \'seed\': seed, \'lr\': lr, \'best_val_group_Dice\': best,\n              \'epochs_completed\': len(history), \'test_evaluated\': False,\n              \'parameters\': sum(p.numel() for p in model.parameters()),\n              \'completed\': len(history) >= epochs or stale >= patience}\n    (dest/\'result.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    return result\n'
(scripts/'colab_runner.py').write_text(runner_source, encoding='utf8')
sys.modules.pop('colab_runner',None)
from colab_runner import MODELS, run, evaluate, partition
MODELS.update(BASES)
seeds=[11,23,37,53,71]
identity=hashlib.sha256((raiz/'package_manifest.json').read_bytes()+codigo.encode()+runner_source.encode()).hexdigest()
for semilla in seeds:
    for red in BASES:
        r=run(red,semilla,salida,epochs=30,lr=0.001,patience=5,identity=identity)
        assert r['completed']

deeplab_compacta 11 {'epoch': 1, 'train_loss': 0.28898261671729863, 'val_group_Dice': 0.654692235175219, 'seconds': 36.18687401700004}
deeplab_compacta 11 {'epoch': 2, 'train_loss': 0.19597997786156063, 'val_group_Dice': 0.6863610156270785, 'seconds': 34.64756174199999}
deeplab_compacta 11 {'epoch': 3, 'train_loss': 0.17438171841387679, 'val_group_Dice': 0.6877997624917315, 'seconds': 34.98869271199999}
deeplab_compacta 11 {'epoch': 4, 'train_loss': 0.1616982248194259, 'val_group_Dice': 0.6926657979950901, 'seconds': 35.555673274000014}
deeplab_compacta 11 {'epoch': 5, 'train_loss': 0.15160709655949195, 'val_group_Dice': 0.7318393800772036, 'seconds': 35.56350456400003}
deeplab_compacta 11 {'epoch': 6, 'train_loss': 0.1371990955668717, 'val_group_Dice': 0.7582644326621623, 'seconds': 35.922371836000025}
deeplab_compacta 11 {'epoch': 7, 'train_loss': 0.13213350193052015, 'val_group_Dice': 0.7311944587711812, 'seconds': 35.96798196300006}
deeplab_compacta 11 {'epoch': 8, 'train_loss': 0.

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


pspnet_compacta 11 {'epoch': 1, 'train_loss': 0.2875152027280313, 'val_group_Dice': 0.6368678282909767, 'seconds': 45.30435858100009}
pspnet_compacta 11 {'epoch': 2, 'train_loss': 0.20113938636894296, 'val_group_Dice': 0.6570171774842956, 'seconds': 43.372320532}
pspnet_compacta 11 {'epoch': 3, 'train_loss': 0.1821229991746213, 'val_group_Dice': 0.6867524920188293, 'seconds': 44.518323793000036}
pspnet_compacta 11 {'epoch': 4, 'train_loss': 0.16952142795832287, 'val_group_Dice': 0.6761881506638245, 'seconds': 43.72764817699999}
pspnet_compacta 11 {'epoch': 5, 'train_loss': 0.16152593639904056, 'val_group_Dice': 0.7039023153609304, 'seconds': 44.780347539999866}
pspnet_compacta 11 {'epoch': 6, 'train_loss': 0.15612825633306515, 'val_group_Dice': 0.7219679348460912, 'seconds': 43.17695744899993}
pspnet_compacta 11 {'epoch': 7, 'train_loss': 0.1490346207113689, 'val_group_Dice': 0.698206067517698, 'seconds': 43.74673904399992}
pspnet_compacta 11 {'epoch': 8, 'train_loss': 0.14496719758186

In [ ]:
# Abrir el test reservado solo con las diez corridas cerradas.
for red in BASES:
    for s in seeds:
        assert json.loads((salida/f'{red}_seed{s}_lr0.001/result.json').read_text())['completed']
extraer('test_v2.zip', 'c324a4a69eed19f78413789a1d557f8a0880975462081ecf96926428b56c68a7')
prep=scripts.parent/'resultados/preentrenamiento'
freeze=json.loads((prep/'freeze_manifest.json').read_text())['artifacts']
for name in ['train.csv','val.csv','test.csv','normalization_train.json']:
    assert hashlib.sha256((prep/name).read_bytes()).hexdigest()==freeze[name]
rows=partition('test',allow_test=True)
assert len(rows)==218 and len({r['group_id'] for r in rows})==122
assert not {r['group_id'] for r in rows} & {r['group_id'] for part in ['train','val'] for r in partition(part)}
detalle=[]
for red in BASES:
    for s in seeds:
        net=BASES[red]().cuda()
        ck=(salida/f'{red}_seed{s}_lr0.001/best.pt')
        net.load_state_dict(torch.load(ck,map_location='cuda',weights_only=True))
        score,detail=evaluate(net,rows,torch.device('cuda'))
        detalle.extend(dict(model=red,seed=s,**r) for r in detail)
        print(red,s,score)
with (salida/'por_imagen.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=detalle[0]); w.writeheader(); w.writerows(detalle)
import shutil
print(shutil.make_archive('/content/IMO_resultados_v2','zip',salida))
